In [1]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
excel_file = pd.ExcelFile("../../../data/bpic19.xlsx", engine="openpyxl")

df = pd.concat(
    [
        pd.read_excel(
            excel_file,
            keep_default_na=False,
            dtype={
                "case:concept:name": "string",
                "concept:name": "string",
                "case:Spend area text": "string",
                "case:Document Type": "string",
                "case:Sub spend area text": "string",
                "case:Purch. Doc. Category name": "string",
                "case:Item Type": "string",
                "case:Item Category": "string",
                "case:Spend classification text": "string",
                "case:Source": "string",
                "case:GR-Based Inv. Verif.": "string",
                "case:Goods Receipt": "string",
                "Cumulative net worth (EUR)": "float32",
                "time_delta": "float32",
            }
        )
        for sheet in excel_file.sheet_names
    ],
    ignore_index=True,
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,Cumulative net worth (EUR),case:Document Type,case:GR-Based Inv. Verif.,case:Goods Receipt,case:Item Category,case:Item Type,case:Purch. Doc. Category name,case:Source,case:Spend area text,case:Spend classification text,case:Sub spend area text,concept:name,time_delta
0,2000000000_00001,2018-01-02 12:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Created,0.0
1,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Complete,3600.0
2,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Awaiting Approval,0.0
3,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Document Completed,0.0
4,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: In Transfer to Execution Syst.,0.0
5,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Ordered,0.0
6,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Change was Transmitted,0.0
7,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Create Purchase Order Item,0.0
8,2000000000_00001,2018-01-02 22:59:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Vendor creates invoice,32760.0
9,2000000000_00001,2018-03-06 06:44:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Record Goods Receipt,5384700.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Cumulative net worth (EUR)', 'case:Document Type', 'case:GR-Based Inv. Verif.', 'case:Goods Receipt', 'case:Item Category', 'case:Item Type', 'case:Purch. Doc. Category name', 'case:Source', 'case:Spend area text', 'case:Spend classification text', 'case:Sub spend area text', 'concept:name', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:Document Type             categorical    case     yes    ['EC Purchase order', 'Framework order', 'Standard PO'] N/A        data_derived        
case:GR-Based Inv. Verif.      categorical    case     yes    ['False', 'True

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [11]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [12]:
engine.parallel_sets

[{'Change Delivery Indicator', 'Change Quantity'},
 {'Cancel Invoice Receipt',
  'Cancel Subsequent Invoice',
  'Clear Invoice',
  'Record Invoice Receipt',
  'Record Subsequent Invoice',
  'Remove Payment Block'},
 {'Cancel Invoice Receipt', 'Cancel Subsequent Invoice'},
 {'Change Currency', 'Change Price', 'Change payment term'},
 {'SRM: Change was Transmitted', 'SRM: Ordered'}]

In [13]:
engine.branching_sets

[{'Block Purchase Order Item',
  'Cancel Goods Receipt',
  'Cancel Invoice Receipt',
  'Cancel Subsequent Invoice',
  'Change Approval for Purchase Order',
  'Change Currency',
  'Change Delivery Indicator',
  'Change Price',
  'Change Quantity',
  'Change Rejection Indicator',
  'Change Storage Location',
  'Change payment term',
  'Clear Invoice',
  'Create Purchase Order Item',
  'Create Purchase Requisition Item',
  'Delete Purchase Order Item',
  'Reactivate Purchase Order Item',
  'Receive Order Confirmation',
  'Record Goods Receipt',
  'Record Invoice Receipt',
  'Record Service Entry Sheet',
  'Record Subsequent Invoice',
  'Release Purchase Order',
  'Remove Payment Block',
  'SRM: Awaiting Approval',
  'SRM: Change was Transmitted',
  'SRM: Complete',
  'SRM: Created',
  'SRM: Document Completed',
  'SRM: Held',
  'SRM: In Transfer to Execution Syst.',
  'SRM: Incomplete',
  'SRM: Ordered',
  'Vendor creates debit memo',
  'Vendor creates invoice'},
 {'Record Invoice Receipt

### --- Experiments Generation ---

In [14]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic19-cf_seed777_experiments_ga_output.txt", console=False)

In [15]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [16]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

In [17]:
exp_df_mul, metadata_mul = ExperimentHandler.load("../experiments/cf_generated_experiments_multiple_desired")
print("Mined using parameters:", metadata_mul["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,4508042805_00010,6,1,2,0.282223,0.279445,0.285,0.332143,0.306667,...,0.376190,0.133333,0.100000,0.2,0.000000,0.142857,0.0,0.931078,0.0,1.0
1,0,4508044035_00320,8,1,2,0.389161,0.398322,0.380,0.450000,0.368421,...,0.437218,0.315789,0.050000,0.1,0.000000,0.071429,0.0,0.000000,0.0,0.0
2,0,4507029240_00300,10,1,4,0.412459,0.474917,0.350,0.478571,0.382609,...,0.612210,0.347826,0.050098,0.1,0.000196,0.214286,0.0,0.928847,0.0,1.0
3,0,4507017655_00360,12,1,6,0.375281,0.475562,0.275,0.382143,0.351852,...,0.539153,0.296296,0.100000,0.2,0.000000,0.142857,0.0,0.921735,0.0,1.0
4,0,4507037320_00020,14,1,6,0.393839,0.537679,0.250,0.392857,0.487097,...,0.516129,0.516129,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,49,4507026467_00001,40,1,0,0.269638,0.324277,0.215,0.443750,0.028916,...,0.137848,0.000000,0.044098,0.0,0.088195,0.093750,0.0,0.000000,0.0,0.0
463,49,4507032796_00001,42,1,0,0.394933,0.379866,0.410,0.604687,0.021839,...,0.095620,0.000000,0.001870,0.0,0.003741,0.093750,0.0,0.000000,0.0,0.0
464,49,4507024372_00001,44,1,0,0.359867,0.389734,0.330,0.557812,0.009890,...,0.047928,0.000000,0.016678,0.0,0.033356,0.031250,0.0,0.000000,0.0,0.0
465,49,4507026083_00001,48,1,0,0.316999,0.313997,0.320,0.501562,0.049495,...,0.068851,0.000000,0.006351,0.0,0.012703,0.062500,0.0,0.000000,0.0,0.0


In [21]:
results_mul = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_multiple_desired_seed777",
    exp_df=exp_df_mul,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [22]:
results_mul

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,4508042805_00010,6,2,2,0.310336,0.360673,0.260,0.333333,0.588889,...,0.887226,0.277778,0.050000,0.1,0.000000,0.083333,0.476115,0.474914,0.500000,0.500000
1,0,4508044035_00320,8,2,2,0.303837,0.357673,0.250,0.308333,0.627273,...,1.585209,0.636364,0.000000,0.0,0.000000,0.000000,0.948846,0.948846,1.000000,1.000000
2,0,4507029240_00300,10,2,4,0.292520,0.370039,0.215,0.308333,0.553846,...,1.229061,0.423077,0.006383,0.0,0.012766,0.083333,0.716268,0.704754,0.500000,0.500000
3,0,4507017655_00360,12,2,6,0.346910,0.423819,0.270,0.350000,0.531667,...,1.009207,0.300000,0.057881,0.1,0.015763,0.166667,0.484659,0.468089,0.500000,0.500000
4,0,4507037320_00020,14,2,6,0.294969,0.294937,0.295,0.362500,0.597059,...,1.114912,0.500000,0.050000,0.1,0.000000,0.083333,0.481578,0.467529,0.500000,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478,49,4507025254_00280,12,4,6,0.347768,0.380536,0.315,0.430556,0.463889,...,0.997861,0.305556,0.171663,0.2,0.143326,0.222222,0.298420,0.891924,0.333333,1.000000
479,49,4507026740_00190,12,4,6,0.299686,0.224373,0.375,0.447222,0.583333,...,1.079009,0.305556,0.196418,0.3,0.092837,0.277778,0.299258,0.889418,0.333333,1.000000
480,49,4507008834_00010,14,4,8,0.366961,0.408922,0.325,0.436111,0.466250,...,1.044657,0.425000,0.000000,0.0,0.000000,0.000000,0.619657,0.910078,0.666667,1.000000
481,49,4507010883_00190,14,4,8,0.380714,0.346427,0.415,0.525000,0.493750,...,0.930675,0.225000,0.163413,0.2,0.126826,0.277778,0.264484,0.889588,0.333333,1.000000


### --- Cleanup ---

In [23]:
sys.stdout = original_stdout
log_file.close()